In [ ]:
# ============================================================
# CELL 1 - Load core + state-aware agent helpers
# ============================================================
exec(open('agent_core.py', encoding='utf-8').read())

from agents import (
    AgentConfig,
    BrowserAgent,
    build_agent_prompt,
    classify_chat_state,
    is_complete as agent_is_complete,
    update_state as agent_update_state,
)

try:
    cfg = http_json("GET", "/api/admin/config")["config"]
    print("Server OK, poll_ms =", cfg.get("poll_ms"))
except Exception as e:
    print("Server NOT reachable:", e)
    print("Run: .venv\\Scripts\\python.exe .\\diagnostic_controller.py")


In [ ]:
# ============================================================
# CELL 2 - Active roles + default task config
# ============================================================
ACTIVE_ROLES = ["DEV", "REVIEW", "AUDIT"]
# ACTIVE_ROLES = ["PLAN", "DEV", "REVIEW", "AUDIT"]
# ACTIVE_ROLES = ["SOLO"]

TIMEOUT_S = 3000
MAX_TURNS = 50
SLEEP_S = 3
STATE_WAIT_S = 3
SEND_MAX_RETRIES_LOCAL = 3
SYSTEM_PROMPT_EVERY_N_ASKS_LOCAL = 5
MAX_STATE_CHARS_LOCAL = 12000

ROLE_ASK_COUNTS.clear()

GOAL = """
Describe the goal here.
""".strip()

CURRENT_TASK = """
Describe the current task here.
""".strip()

log_roles_status(ACTIVE_ROLES)


In [ ]:
# ============================================================
# CELL 3 - Multi-agent helpers (all sends go through agents.py)
# ============================================================
import time


def make_browser_agent(role, active_roles=None, timeout_s=TIMEOUT_S):
    active = [r.upper().strip() for r in (active_roles or ACTIVE_ROLES)]
    config = AgentConfig(
        role=role,
        active_roles=active,
        timeout_s=timeout_s,
        sleep_s=SLEEP_S,
        state_wait_s=STATE_WAIT_S,
        state_reload_after_errors=3,
        send_max_retries=SEND_MAX_RETRIES_LOCAL,
        max_state_chars=MAX_STATE_CHARS_LOCAL,
        system_prompt_every_n_asks=SYSTEM_PROMPT_EVERY_N_ASKS_LOCAL,
    )
    return BrowserAgent(
        config,
        run_command_fn=run_command,
        http_json_fn=http_json,
        try_reset_page_fn=try_reset_page,
        parse_routing_fn=parse_routing,
        sync_timeout_s=SYNC_TIMEOUT_S,
        probe_timeout_s=PROBE_TIMEOUT_S,
        set_prompt_timeout_s=SET_PROMPT_TIMEOUT_S,
        click_timeout_s=CLICK_TIMEOUT_S,
    )


def should_attach_system(role):
    asks = ROLE_ASK_COUNTS.get(role, 0)
    return (asks % SYSTEM_PROMPT_EVERY_N_ASKS_LOCAL) == 0


def ask_role(role, goal, state, turn, *, active_roles=None, stale_response="", attach_system=None, extra_prompt=""):
    role = role.upper().strip()
    active = [r.upper().strip() for r in (active_roles or ACTIVE_ROLES)]
    agent = make_browser_agent(role, active_roles=active)
    attach = should_attach_system(role) if attach_system is None else bool(attach_system)
    prompt_base = load_prompt(role) if attach else ""
    prompt = build_agent_prompt(prompt_base, goal, state, turn, agent.config, attach_system=attach)
    if extra_prompt:
        prompt = f"{prompt}\n\n{extra_prompt.strip()}"

    if attach:
        print(f"[prompt] attach {role} system prompt at ask #{ROLE_ASK_COUNTS.get(role, 0) + 1}")
    response = agent.send_and_wait(prompt, stale_response=stale_response)
    ROLE_ASK_COUNTS[role] = ROLE_ASK_COUNTS.get(role, 0) + 1
    return response


def parse_target(response):
    if agent_is_complete(response):
        return "TASK COMPLETE", response
    routing = parse_routing(response)
    if not routing:
        return "", response
    target = normalize_routing_target(routing.get("target", ""))
    message = str(routing.get("message") or response).strip()
    return target, message


def wait_or_resume_role(role):
    agent = make_browser_agent(role)
    state = agent.wait_for_sendable_chat()
    if state.get("should_wait_response"):
        return agent.wait_for_live_response()
    if state.get("kind") == "assistant_ready":
        return state.get("response", "")
    return ""


def run_routed_agent_loop(goal, *, start_role=None, active_roles=None, max_turns=MAX_TURNS):
    active = [r.upper().strip() for r in (active_roles or ACTIVE_ROLES)]
    current_role = (start_role or active[0]).upper().strip()
    history = []
    state = f"GOAL:\n{goal}"

    for turn in range(1, max_turns + 1):
        print(f"\n{'=' * 60}\n{current_role} | TURN {turn}\n{'=' * 60}")
        stale = history[-1][1] if history else ""
        response = ask_role(current_role, goal, state, turn, active_roles=active, stale_response=stale)
        print(response[:1200] + ("..." if len(response) > 1200 else ""))
        history.append((current_role, response))

        target, message = parse_target(response)
        if target in {"TASK COMPLETE", "FINISH"}:
            print("\n[TASK COMPLETE]")
            return {"status": "complete", "history": history, "last_response": response}

        routing = parse_routing(response)
        state = agent_update_state(state, response, routing, turn, AgentConfig(current_role, active))

        if target in active:
            print(f"[routing] {current_role} -> {target}")
            current_role = target
        else:
            print(f"[routing] no valid target; staying on {current_role}")
            state += f"\n\nROUTING_NOTE:\nNo valid target was found. Continue as {current_role}."
        time.sleep(SLEEP_S)

    print(f"\n[max_turns] reached {max_turns}")
    return {"status": "max_turns", "history": history, "last_response": history[-1][1] if history else ""}


In [ ]:
# ============================================================
# CELL 4 - Run generic routed multi-agent loop
# ============================================================
ACTIVE_ROLES = ["DEV", "REVIEW", "AUDIT"]
ROLE_ASK_COUNTS.clear()

GOAL = f"""
{CURRENT_TASK}
""".strip()

result_multi = run_routed_agent_loop(
    GOAL,
    start_role=ACTIVE_ROLES[0],
    active_roles=ACTIVE_ROLES,
    max_turns=MAX_TURNS,
)

print("\n[result]", result_multi["status"], "turns=", len(result_multi["history"]))


In [ ]:
# ============================================================
# CELL 5 - C/D/T flow using the same state-aware agent runner
# ============================================================
ACTIVE_ROLES = ["C", "D", "T"]
ROLE_ASK_COUNTS.clear()

GOAL = f"Dat goal qua cac plan nho co review: {CURRENT_TASK}"
MAX_PLAN_CYCLES = 10
MAX_ROUNDS_PLAN = 8

T_EXTRA = """
You are T (Task Planner). Create a short executable plan.
If D says TASK COMPLETE, answer GOAL COMPLETE when the full goal is done, otherwise NEXT PLAN.
""".strip()

C_EXTRA = """
You are C (Developer). Execute the current PLAN. End with READY_FOR_REVIEW when ready.
""".strip()

D_EXTRA = """
You are D (Reviewer/Tester). Review C output against PLAN. First line must be TASK COMPLETE, CHANGES REQUESTED, or NEED INFO.
""".strip()

history_cdt = []
plan = ask_role("T", GOAL, f"CURRENT_TASK:\n{CURRENT_TASK}", 1, active_roles=ACTIVE_ROLES, extra_prompt=T_EXTRA)
history_cdt.append(("T", plan))
print(f"\n===== T PLAN 1 =====\n{plan}\n")

for cycle in range(1, MAX_PLAN_CYCLES + 1):
    c_out = d_out = ""
    for round_index in range(1, MAX_ROUNDS_PLAN + 1):
        c_state = f"CURRENT_TASK:\n{CURRENT_TASK}\n\nPLAN:\n{plan}"
        if d_out:
            c_state += f"\n\nREVIEW_FROM_D:\n{d_out}"
        c_out = ask_role("C", GOAL, c_state, round_index, active_roles=ACTIVE_ROLES, stale_response=c_out, extra_prompt=C_EXTRA)
        history_cdt.append(("C", c_out))
        print(f"\n===== C PLAN {cycle} ROUND {round_index} =====\n{c_out}\n")
        time.sleep(SLEEP_S)

        d_state = f"CURRENT_TASK:\n{CURRENT_TASK}\n\nPLAN:\n{plan}\n\nOUTPUT_C:\n{c_out}"
        d_out = ask_role("D", GOAL, d_state, round_index, active_roles=ACTIVE_ROLES, stale_response=d_out, extra_prompt=D_EXTRA)
        history_cdt.append(("D", d_out))
        print(f"\n===== D PLAN {cycle} ROUND {round_index} =====\n{d_out}\n")
        if is_task_complete(d_out):
            break
        time.sleep(SLEEP_S)

    t_state = f"CURRENT_TASK:\n{CURRENT_TASK}\n\nPLAN:\n{plan}\n\nC:\n{c_out}\n\nD:\n{d_out}"
    t_out = ask_role("T", GOAL, t_state, cycle + 1, active_roles=ACTIVE_ROLES, stale_response=plan, extra_prompt=T_EXTRA)
    history_cdt.append(("T", t_out))
    print(f"\n===== T DECISION {cycle} =====\n{t_out}\n")
    if is_goal_complete(t_out) or agent_is_complete(t_out):
        print("\n[CDT] GOAL COMPLETE")
        break
    plan = t_out
    time.sleep(SLEEP_S)

print(f"\n[CDT] History: {len(history_cdt)} turns logged")


In [ ]:
# ============================================================
# CELL 6 - SOLO flow through agents.py (same behavior as solo.py)
# ============================================================
ACTIVE_ROLES = ["SOLO"]
ROLE_ASK_COUNTS.clear()

GOAL = """
Complete the task end-to-end with self-verification.
""".strip()

CURRENT_TASK = """
Do task, test it, and report completion when done.
""".strip()

result_solo = run_routed_agent_loop(
    f"{GOAL}\n\nCURRENT_TASK:\n{CURRENT_TASK}",
    start_role="SOLO",
    active_roles=ACTIVE_ROLES,
    max_turns=MAX_TURNS,
)

print("\n[solo result]", result_solo["status"], "turns=", len(result_solo["history"]))
